## Introduction

The unionTables function provides a high-performance alternative to standard Spark SQL set operations like `INTERSECT`, `EXCEPT`, and `UNION`. Traditional set operations are **wide transformations** that require shuffling entire rows of data across the cluster's nodes. This process can be a significant bottleneck, especially for large datasets on clusters with limited network bandwidth, potentially causing out-of-space conditions.

To mitigate this, the function avoids comparing and shuffling full data rows. Instead, it computes a **lightweight and fast `xxhash64` value for each row**. By assuming rows with identical hashes as identical, it performs all comparisons and joins on these small integer hashes rather than the potentially large and complex row data.

The function returns a single DataFrame containing the distinct union of all input tables. Crucially, it adds a boolean **"indicator" column** for each input DataFrame (e.g., `__unionTables_in_tableA`). A row will have `True` in this column if it was present in the corresponding input table. From this single output, users can efficiently derive `INTERSECT`, `EXCEPT`, or `UNION` results by simply filtering on these indicator columns, drastically reducing data shuffling and possibly improving execution time.

**Warning**: there is a small ( ~ num_of_rows/2^64 , as a 64-bit hash is used) but not non-zero probability of hash collision of two different rows, which will cause one of the rows to be lost in the result. Additional computation to assure non-collision is needed to guarantee exactness.

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import xxhash64, col, lit
from functools import reduce

def unionTables(tables: dict[str, DataFrame]):
    # tables map name to DataFrame
    # returns a dataframe containing a distinct union of all DataFrames in tables, plus bool columns [f'__unionTables_in_{name}' for name in tables.keys()]
    # __unionTables_in_{name} value be true if row exists by value in DataFrame 'name', false if it does not
    
    if not tables:
        return None

    # first, if any dataframe schema in tables are different from each other, raise exception.
    table_iter = iter(tables.values())
    first_schema = next(table_iter).schema
    if not all(df.schema == first_schema for df in table_iter):
        raise ValueError("All DataFrames must have the same schema.")

    # we want to use row hashing to avoid expensive data shuffling among different dataframes, which could be stored in different nodes.
    # for each DataFrame, hash all columns and store into a new column called __unionTables_hash using pyspark.sql.functions.xxhash64
    # assume two rows are the same if their xxhash64 are the same, ignore the tiny probability of two different rows have hash conflict.
    
    hashed_tables = {}
    for name, df in tables.items():
        hashed_tables[name] = df.withColumn("__unionTables_hash", xxhash64(*df.columns))

    # Union and Deduplication
    all_hashed_dfs = list(hashed_tables.values())
    if len(all_hashed_dfs) == 1:
        unioned_df = all_hashed_dfs[0]
    else:
        unioned_df = reduce(lambda df1, df2: df1.union(df2), all_hashed_dfs) # pySpark union is like SQL union all
    
    distinct_df = unioned_df.dropDuplicates(['__unionTables_hash'])

    # Indicator Columns
    result_df = distinct_df
    for name, hashed_df in hashed_tables.items():
        indicator_col = f'__unionTables_in_{name}'
        # Create a dataframe with just the hash and a literal true value
        indicator_df = hashed_df.select(col("__unionTables_hash").alias(f"hash_{name}"), lit(True).alias(indicator_col))
        # Left join to add the indicator column
        result_df = result_df.join(indicator_df, result_df["__unionTables_hash"] == indicator_df[f"hash_{name}"], "left").drop(f"hash_{name}")

    # Final Touches
    for name in tables.keys():
        indicator_col = f'__unionTables_in_{name}'
        result_df = result_df.fillna(False, subset=[indicator_col])

    return result_df.drop('__unionTables_hash')


## Example

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import random
from datetime import datetime, date
random.seed(42)

spark = SparkSession.builder \
    .appName("SingleTaskApp") \
    .master("local[1]") \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true') \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
data1 = [("a", 1), ("b", 2), ("c", 3)]
data2 = [("b", 2), ("c", 3), ("d", 4)]
data3 = [("c", 3), ("d", 4), ("e", 5)]

columns = ["col1", "col2"]

df1 = spark.createDataFrame(data1, columns)
df2 = spark.createDataFrame(data2, columns)
df3 = spark.createDataFrame(data3, columns)

tables = {
    "table1": df1,
    "table2": df2,
    "table3": df3
}

result = unionTables(tables)
result.show()

# Expected output:
# +----+----+-----------------------+-----------------------+-----------------------+
# |col1|col2|__compareTable_in_table1|__compareTable_in_table2|__compareTable_in_table3|
# +----+----+-----------------------+-----------------------+-----------------------+
# |   a|   1|                   true|                  false|                  false|
# |   b|   2|                   true|                   true|                  false|
# |   c|   3|                   true|                   true|                   true|
# |   d|   4|                  false|                   true|                   true|
# |   e|   5|                  false|                  false|                   true|
# +----+----+-----------------------+-----------------------+-----------------------+


## Little benchmark
### Create random test data

In [ ]:
schema = StructType([
    StructField("integer_col", IntegerType(), True),
    StructField("double_col", DoubleType(), True),
    StructField("string_col", StringType(), True),
    StructField("boolean_col", BooleanType(), True),
    StructField("timestamp_col", TimestampType(), True),
    StructField("date_col", DateType(), True),
    StructField("array_col", ArrayType(IntegerType()), True),
    StructField("struct_col", StructType([
        StructField("nested_int", IntegerType(), True),
        StructField("nested_string", StringType(), True)
    ]), True)
])

num_rows = 2000000
data = []
for _ in range(num_rows):
    row = (
        random.randint(0, 1000),
        random.uniform(0, 1000),
        'string_' + str(random.randint(0, 100)),
        random.choice([True, False]),
        datetime.fromtimestamp(random.randint(0, 1672531199)),
        date.fromordinal(random.randint(1, 738140)),
        [random.randint(0, 10) for _ in range(3)],
        (random.randint(0, 10), 'nested_' + str(random.randint(0, 10)))
    )
    data.append(row)

df = spark.createDataFrame(data, schema)

# Save the DataFrame to a Parquet file
df.write.mode('overwrite').parquet("2million_a.parquet")

df.write.mode('overwrite').parquet("2million_b.parquet")

### Test speed

In [2]:
import time

# Load the datasets
a = spark.read.parquet("2million_a.parquet")
b = spark.read.parquet("2million_b.parquet")

# --- Time unionTables ---
start_time = time.time()
result_union = unionTables({'a': a, 'b': b})
a_except_b_union = result_union.filter("`__unionTables_in_a` = True AND `__unionTables_in_b` = False")
count_union = a_except_b_union.count()
end_time = time.time()
print(f"unionTables execution time: {end_time - start_time:.2f} seconds")
print(f"Count of A EXCEPT B (unionTables): {count_union}")

# --- Time exceptAll ---
start_time = time.time()
a_except_b_native = a.exceptAll(b)
count_native = a_except_b_native.count()
end_time = time.time()
print(f"exceptAll execution time: {end_time - start_time:.2f} seconds")
print(f"Count of A EXCEPT B (exceptAll): {count_native}")

unionTables execution time: 9.83 seconds
Count of A EXCEPT B (unionTables): 0


exceptAll execution time: 7.19 seconds
Count of A EXCEPT B (exceptAll): 0
